# 03 · Forgery Detection (contest exercise)

**Style C: contest simulation**, a close copy of T25-ANTIQUE ("Antique Painting Authentication", IOAI 2025
Day 2): tabular features, labels `1` / `-1` / `0 = unknown`, headerless A/B CSVs in a zip, accuracy.
**Traces to** `notes/ioai-task-patterns.md` P1, P2, P7.

## 1. Problem Description

A museum digitised 600 paintings into 6 numeric features (pigment ratios, canvas density, …). Experts
authenticated only a handful of them, because authentication is expensive. The features are known to have strong
structure. Classify the paintings as authentic (`1`) or replica (`-1`), using the unlabelled paintings as well.

## 2. Dataset

| Split | Rows | Labels |
|---|---|---|
| `train` | 600 | column `label`: `1` authentic, `-1` replica, `0` **unknown** (most rows) |
| `test_a` | 600 | hidden (Leaderboard A) |
| `test_b` | 600 | hidden (Leaderboard B, **final**) |

The 6 feature columns `f0..f5` are on very different scales.

## 3. Task

Predict `1` or `-1` for every row of `test_a` and `test_b`.

## 4. Submission

`submission.zip` with `submissionA.csv` and `submissionB.csv`: **one value (`1` or `-1`) per line, no
header**, in row order.

## 5. Scoring

Accuracy.
- Baseline (Leaderboard B): **0.48** (SVC after filling unknown labels at random, as in the T25 baseline)
- Reference (Leaderboard B): **0.97**, which is your target. T25-ANTIQUE's SC score was 0.98.

## 6. Requirements

- ≤ 2 minutes on CPU, scikit-learn / numpy / scipy only.
- The test features may be used unlabelled (transductively), as in the real task.

In [ ]:
import time, os, zipfile, base64, zlib, random
T0 = time.time()
import numpy as np, pandas as pd
seed = 42
random.seed(seed); np.random.seed(seed)

### Data loading (DO NOT CHANGE, DO NOT DECODE)

In [ ]:
_BLOB = ("eNpNkdFOgzAUhu/3FOeyaNcU5qYSSXyELfGChBDSuG42oy22JRMvfHYPLcYRAofT7///Q6v0YF0AM+phAuHBDKujPEE3CGWCMmdPsnIFeJ2c1eAvvRTOsKMIwsvgQSW5FhfZaWuNj6wzZ6jQijlhjlYzdBRjHzrsk4IX2yxSewoTYv9aYjov9NBLX+VPnFMwVnlZccafKSSrzgcRZPWYHA4UuhTUKyP6M/t0BDOYsU6Lnnj1LSuyo7DLsiSokd7DKxyakkJZtOwN7uFGgZmYhookTVPsbsQ13M1xwjkxkSZnuPzA52cssSqWcst4m6H5PxyxDY+vdZHwLdJ8UbYpZUo/dP2QThL8qCCn873O07o6fiExzzxIp0fcDmVNnHSZMu1q3SDYYh3fybkLDk8VF6em3HHesnc7TCTJLsZezWKctpogksHLPNrmVt/8RLZFmKfTlmF0BhOjKf3jKHawUeZF7NbNXJRxpJv2tLRXv/RutEI=")
exec(zlib.decompress(base64.b64decode(_BLOB)).decode())
_Xtr, _ytr, _Xa, _Xb, _, _ = _paintings()
COLS = [f"f{i}" for i in range(6)]
train = pd.DataFrame(_Xtr, columns=COLS); train["label"] = _ytr
test_a = pd.DataFrame(_Xa, columns=COLS)
test_b = pd.DataFrame(_Xb, columns=COLS)
print(train["label"].value_counts().to_dict())
train.describe().round(2)

### Baseline (replace this)

The official T25 baseline: fill the unknown labels at random, then fit an RBF SVC on the raw features.

In [ ]:
from sklearn.svm import SVC

def fit_predict(train_df, test_dfs):
    X = train_df[COLS].to_numpy(); y = train_df["label"].to_numpy().copy()
    rng = np.random.default_rng(42)
    y[y == 0] = rng.choice([-1, 1], size=(y == 0).sum())
    model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42).fit(X, y)
    return [model.predict(t[COLS].to_numpy()) for t in test_dfs]

### Local validation

Only the labelled rows can be scored, so we use leave-some-labels-out: hide half of the known labels and check
how they are predicted.

In [ ]:
known_idx = np.flatnonzero(train["label"].to_numpy() != 0)
hide = np.random.default_rng(0).permutation(known_idx)[: len(known_idx) // 2]
masked = train.copy(); masked.loc[hide, "label"] = 0
(val_pred,) = fit_predict(masked, [train.loc[hide]])
print(f"accuracy on {len(hide)} hidden labels: {(val_pred == train.loc[hide, 'label'].to_numpy()).mean():.3f}")

### Submission

In [ ]:
pred_a, pred_b = fit_predict(train, [test_a, test_b])
for name, p in [("submissionA.csv", pred_a), ("submissionB.csv", pred_b)]:
    pd.Series(np.asarray(p).astype(int)).to_csv(name, index=False, header=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submissionA.csv"); z.write("submissionB.csv")
with zipfile.ZipFile("submission.zip") as z:
    for name in ["submissionA.csv", "submissionB.csv"]:
        lines = z.read(name).decode().splitlines()
        assert len(lines) == 600 and set(lines) <= {"1", "-1"}, f"{name} malformed"
print("submission.zip written and format-checked")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    ya, yb = _paintings()[4:]
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, y in [("submissionA.csv", ya), ("submissionB.csv", yb)]:
            try:
                p = np.array([int(v) for v in z.read(name).decode().splitlines()])
                out.append((p == y).mean() if len(p) == len(y) else 0.0)
            except Exception:
                out.append(0.0)
    return out

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
print(f"total runtime {(time.time() - T0) / 60:.2f} min (limit 2 min)")